In [1]:
import heapq
from itertools import count
import time

class Block_Puzzle:
    def __init__(self, state):
        self.state = state
        self.size = int(len(state) ** 0.5)

    def __eq__(self, other):
        return self.state == other.state

    def __hash__(self):
        return hash(tuple(self.state))

    def __str__(self):
        s = ''
        for i in range(0, len(self.state), self.size):
            s += ' '.join(map(str, self.state[i:i+self.size])) + '\n'
        return s

    def get_neighbors(self):
        neighbors = []
        zero_index = self.state.index(0)
        row, col = divmod(zero_index, self.size)

        directions = [(-1,0), (1,0), (0,-1), (0,1)]  # up, down, left, right

        for dr, dc in directions:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < self.size and 0 <= new_col < self.size:
                new_index = new_row * self.size + new_col
                new_state = self.state[:]
                new_state[zero_index], new_state[new_index] = new_state[new_index], new_state[zero_index]
                neighbors.append(Block_Puzzle(new_state))
        return neighbors

    def heuristic_estimate_misplaced_tiles(self, goal):
        return sum(1 for i, val in enumerate(self.state) if val != 0 and val != goal.state[i])

class AASTERISK:
    def __init__(self):
        self.counter = count()

    def run_Astar(self, start, goal):
        openSet = []
        heapq.heappush(openSet, (0, next(self.counter), start))
        cameFrom = {}
        gScore = {start: 0}
        fScore = {start: start.heuristic_estimate_misplaced_tiles(goal)}

        while openSet:
            _, _, current = heapq.heappop(openSet)

            if current == goal:
                return self.reconstruct_path(cameFrom, current)

            for neighbor in current.get_neighbors():
                tentative_gScore = gScore[current] + 1

                if neighbor not in gScore or tentative_gScore < gScore[neighbor]:
                    cameFrom[neighbor] = current
                    gScore[neighbor] = tentative_gScore
                    fScore[neighbor] = tentative_gScore + neighbor.heuristic_estimate_misplaced_tiles(goal)
                    heapq.heappush(openSet, (fScore[neighbor], next(self.counter), neighbor))

        return []

    def reconstruct_path(self, cameFrom, current):
        total_path = [current]
        while current in cameFrom:
            current = cameFrom[current]
            total_path.append(current)
        return total_path[::-1]

# Dữ liệu bài toán 8-puzzle
start_8 = Block_Puzzle([
    1, 8, 2,
    0, 4, 3,
    7, 6, 5])

goal_8 = Block_Puzzle([
    1, 2, 3,
    4, 5, 6,
    7, 8, 0])

solver = AASTERISK()

start_time = time.time()
path = solver.run_Astar(start_8, goal_8)
end_time = time.time()

# In kết quả
print("Solution path for 8-puzzle:")
for step_num, step in enumerate(path):
    print(f"Step {step_num}:\n{step}")

print(f"Total moves: {len(path) - 1}")
print(f"Execution time: {end_time - start_time:.4f} seconds")


Solution path for 8-puzzle:
Step 0:
1 8 2
0 4 3
7 6 5

Step 1:
1 8 2
4 0 3
7 6 5

Step 2:
1 0 2
4 8 3
7 6 5

Step 3:
1 2 0
4 8 3
7 6 5

Step 4:
1 2 3
4 8 0
7 6 5

Step 5:
1 2 3
4 8 5
7 6 0

Step 6:
1 2 3
4 8 5
7 0 6

Step 7:
1 2 3
4 0 5
7 8 6

Step 8:
1 2 3
4 5 0
7 8 6

Step 9:
1 2 3
4 5 6
7 8 0

Total moves: 9
Execution time: 0.0004 seconds


In [2]:
import heapq
import time

class NPuzzleSolver:
    def __init__(self, size=3, goal_state=None):
        self.size = size
        self.n = size * size
        if goal_state is None:
            self.goal_state = [i % self.n for i in range(1, self.n + 1)]
        else:
            self.goal_state = goal_state

    def heuristic(self, state):
        """Số ô sai vị trí so với goal_state"""
        misplaced = 0
        for i in range(self.size):
            for j in range(self.size):
                val = state[i][j]
                goal_val = self.goal_state[i * self.size + j]
                if val != 0 and val != goal_val:
                    misplaced += 1
        return misplaced

    def find_zero(self, state):
        for i in range(self.size):
            for j in range(self.size):
                if state[i][j] == 0:
                    return i, j

    def get_neighbors(self, state):
        neighbors = []
        x, y = self.find_zero(state)
        directions = [(-1,0), (1,0), (0,-1), (0,1)]
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.size and 0 <= ny < self.size:
                new_state = [row[:] for row in state]
                new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
                neighbors.append(new_state)
        return neighbors

    def state_to_tuple(self, state):
        return tuple(tuple(row) for row in state)

    def print_state(self, state):
        for row in state:
            print(' '.join(str(num).rjust(2) if num != 0 else '  ' for num in row))
        print()

    def is_goal(self, state):
        flat = [num for row in state for num in row]
        return flat == self.goal_state

    def solve(self, start_state):
        """Chạy A* để giải bài toán"""
        start_time = time.time()
        pq = []
        visited = set()
        heapq.heappush(pq, (self.heuristic(start_state), 0, start_state, []))  # (f, g, state, path)

        while pq:
            f, g, current, path = heapq.heappop(pq)
            state_key = self.state_to_tuple(current)

            if self.is_goal(current):
                total_time = time.time() - start_time
                print("✅ Đã tìm thấy lời giải sau", g, "bước:")
                for idx, s in enumerate(path + [current]):
                    print(f"Step {idx}:")
                    self.print_state(s)
                print(f"🕒 Thời gian chạy: {total_time:.4f} giây")
                return

            if state_key in visited:
                continue
            visited.add(state_key)

            for neighbor in self.get_neighbors(current):
                if self.state_to_tuple(neighbor) not in visited:
                    h = self.heuristic(neighbor)
                    heapq.heappush(pq, (g + 1 + h, g + 1, neighbor, path + [current]))

        print("❌ Không tìm được lời giải.")
start_state = [
    [1, 8, 2],
    [0, 4, 3],
    [7, 6, 5]
]


solver = NPuzzleSolver(size=3)
solver.solve(start_state)


✅ Đã tìm thấy lời giải sau 9 bước:
Step 0:
 1  8  2
    4  3
 7  6  5

Step 1:
 1  8  2
 4     3
 7  6  5

Step 2:
 1     2
 4  8  3
 7  6  5

Step 3:
 1  2   
 4  8  3
 7  6  5

Step 4:
 1  2  3
 4  8   
 7  6  5

Step 5:
 1  2  3
 4  8  5
 7  6   

Step 6:
 1  2  3
 4  8  5
 7     6

Step 7:
 1  2  3
 4     5
 7  8  6

Step 8:
 1  2  3
 4  5   
 7  8  6

Step 9:
 1  2  3
 4  5  6
 7  8   

🕒 Thời gian chạy: 0.0005 giây
